In [1]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3" 

import logging
import tensorflow as tf
tf.get_logger().setLevel(logging.ERROR)

import absl.logging
absl.logging.set_verbosity(absl.logging.ERROR)

import shutil
import random
from pathlib import Path

import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
from sklearn.metrics import classification_report, confusion_matrix

C:\Users\FA706\miniconda3\envs\tf210\lib\site-packages\requests\__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


In [2]:
import numpy
print(numpy.__version__)

1.24.3


In [3]:
import tensorflow as tf
print(tf.__version__)                              # doit afficher 2.10.x
print(tf.config.list_physical_devices('GPU'))      # doit lister ta RTX 3060

2.13.0
[]


In [4]:
SOURCE_DIR  = Path("Tumordataset-2")
WORK_ROOT   = Path("dataset_split-2") 
CLASS_NAMES = ["glioma", "meningioma", "notumor", "pituitary"] 

SPLIT    = {"train": 0.80, "valid": 0.01, "test": 0.19} 
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}

IMG_SIZE   = (224, 224)
BATCH_SIZE = 64
EPOCHS     = 8
SEED       = 42

In [5]:
def split_dataset():
    if WORK_ROOT.exists():
        print("skip,", WORK_ROOT, "existe deja.")
        return
    rng = random.Random(SEED)
    for cls in CLASS_NAMES:
        src = SOURCE_DIR / cls
        files = [p for p in src.iterdir() if p.suffix.lower() in IMG_EXTS]
        rng.shuffle(files)
        n = len(files)
        n_train = int(n * SPLIT["train"])
        n_valid = int(n * SPLIT["valid"])
        buckets = {
            "train": files[:n_train],
            "valid": files[n_train:n_train + n_valid],
            "test":  files[n_train + n_valid:],
        }
        for split, items in buckets.items():
            out = WORK_ROOT / split / cls
            out.mkdir(parents=True, exist_ok=True)
            for p in items:
                shutil.copy(p, out / p.name)
        print(cls, "->", n, "images :", {k: len(v) for k, v in buckets.items()})
    print("Decoupage termine sous", WORK_ROOT)


split_dataset()

skip, dataset_split-2 existe deja.


In [6]:
def load_split(name, shuffle):
    return tf.keras.utils.image_dataset_from_directory(
        WORK_ROOT / name,
        labels="inferred",
        label_mode="int",
        class_names=CLASS_NAMES,
        color_mode="rgb",
        image_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
        seed=SEED,
    )

train_ds = load_split("train", shuffle=True)
val_ds   = load_split("valid", shuffle=False)
test_ds  = load_split("test",  shuffle=False)

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(AUTOTUNE)
val_ds   = val_ds.prefetch(AUTOTUNE)
test_ds  = test_ds.prefetch(AUTOTUNE)

Found 11377 files belonging to 4 classes.
Found 141 files belonging to 4 classes.
Found 2705 files belonging to 4 classes.


In [7]:
data_augmentation = models.Sequential([
    layers.RandomRotation(0.05),
    layers.RandomZoom(0.1),
    layers.RandomContrast(0.1),
], name="augmentation")

In [8]:
def build_cnn(input_shape=(*IMG_SIZE, 3), num_classes=len(CLASS_NAMES)):
    inputs = layers.Input(shape=input_shape)

    x = data_augmentation(inputs)
    x = layers.Rescaling(1.0 / 255)(x)

    # 32 filtre
    x = layers.Conv2D(32, 3, padding="same", activation="relu")(x)
    x = layers.BatchNormalization(momentum=0.9)(x)
    x = layers.MaxPooling2D()(x)

    # 64 filtres
    x = layers.Conv2D(64, 3, padding="same", activation="relu")(x)
    x = layers.BatchNormalization(momentum=0.9)(x)
    x = layers.MaxPooling2D()(x)

    # 128 filtre
    x = layers.Conv2D(128, 3, padding="same", activation="relu")(x)
    x = layers.BatchNormalization(momentum=0.9)(x)
    x = layers.MaxPooling2D()(x)

    # 128 filtres
    x = layers.Conv2D(128, 3, padding="same", activation="relu")(x)
    x = layers.BatchNormalization(momentum=0.9)(x)
    x = layers.MaxPooling2D()(x)

    # Classification
    x = layers.GlobalAveragePooling2D()(x)    
    x = layers.Dropout(0.4)(x)
    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)

    return models.Model(inputs, outputs, name="cnn_from_scratch")


model = build_cnn()
model.summary()

Model: "cnn_from_scratch"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_1 (InputLayer)        [(None, 224, 224, 3)]     0         
                                                                 
 augmentation (Sequential)   (None, 224, 224, 3)       0         
                                                                 
 rescaling (Rescaling)       (None, 224, 224, 3)       0         
                                                                 
 conv2d (Conv2D)             (None, 224, 224, 32)      896       
                                                                 
 batch_normalization (Batch  (None, 224, 224, 32)      128       
 Normalization)                                                  
                                                                 
 max_pooling2d (MaxPooling2  (None, 112, 112, 32)      0         
 D)                                               

In [9]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

cb = [
    callbacks.EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True),
    callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=4, min_lr=1e-6),
    callbacks.ModelCheckpoint("best_cnn.keras", monitor="val_accuracy", save_best_only=True),
]

In [10]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=cb,
)

Epoch 1/8



KeyboardInterrupt



In [ ]:
test_loss, test_acc = model.evaluate(test_ds)
print("Test accuracy :", round(test_acc, 3))

y_true, y_pred = [], []
for images, labels in test_ds:
    probs = model.predict(images, verbose=0)
    y_pred.extend(np.argmax(probs, axis=1))
    y_true.extend(labels.numpy())

print()
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES))

In [ ]:
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(6, 5))
sns.heatmap(
    cm, annot=True, fmt="d", cmap="Blues",
    xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
    cbar=False,
)
plt.xlabel("Prediction")
plt.ylabel("Verite terrain")
plt.title("Matrice de confusion - test set")
plt.tight_layout()
plt.show()

In [ ]:
from tensorflow.keras import Model
from sklearn.metrics.pairwise import cosine_similarity

gap_layer = None
for l in model.layers:
    if "global_average_pooling" in l.name:
        gap_layer = l
        break

feature_extractor = Model(inputs=model.input, outputs=gap_layer.output)
print("Embedding de dimension :", gap_layer.output.shape[-1])

# 2) On extrait l'embedding de toutes les images du test set
embeddings, labels_all = [], []
for images, labels in test_ds:
    feats = feature_extractor.predict(images, verbose=0)   # augmentation inactive en predict
    embeddings.append(feats)
    labels_all.append(labels.numpy())

embeddings = np.concatenate(embeddings)
labels_all = np.concatenate(labels_all)

# 3) Vecteur moyen (centroide) de chaque classe
centroids = np.stack([
    embeddings[labels_all == c].mean(axis=0)
    for c in range(len(CLASS_NAMES))
])

S = cosine_similarity(centroids)

plt.figure(figsize=(6, 5))
sns.heatmap(S, annot=True, fmt=".3f", cmap="viridis",
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
            vmin=0, vmax=1)
plt.title("Similarite cosinus des embeddings CNN - test set")
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.manifold import TSNE

tsne = TSNE(
    n_components=2,
    perplexity=30,
    init="pca",
    learning_rate="auto",
    random_state=42,
    verbose=1,
)
proj = tsne.fit_transform(embeddings) 

plt.figure(figsize=(9, 7))
palette = sns.color_palette("tab10", len(CLASS_NAMES))
for c in range(len(CLASS_NAMES)):
    mask = labels_all == c
    plt.scatter(
        proj[mask, 0], proj[mask, 1],
        s=12, alpha=0.6, color=palette[c], label=CLASS_NAMES[c],
    )

plt.legend(title="Classe reelle", markerscale=2)
plt.title("Projection t-SNE")
plt.xlabel("")
plt.ylabel("")
plt.tight_layout()
plt.show()

In [ ]:
hist = history.history
epochs_range = list(range(1, len(hist["loss"]) + 1))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

sns.lineplot(x=epochs_range, y=hist["accuracy"], label="train", ax=ax1)
sns.lineplot(x=epochs_range, y=hist["val_accuracy"], label="val", ax=ax1)
ax1.set_title("Accuracy")
ax1.set_xlabel("epoch")

sns.lineplot(x=epochs_range, y=hist["loss"], label="train", ax=ax2)
sns.lineplot(x=epochs_range, y=hist["val_loss"], label="val", ax=ax2)
ax2.set_title("Loss")
ax2.set_xlabel("epoch")

plt.tight_layout()
plt.show()

In [ ]:
PER_CLASS = 2

buckets = {c: [] for c in range(len(CLASS_NAMES))}
for images, labels in test_ds:
    for img, lab in zip(images, labels):
        lab = int(lab.numpy())
        if len(buckets[lab]) < PER_CLASS:
            buckets[lab].append(img)
    if all(len(v) >= PER_CLASS for v in buckets.values()):
        break

sel_images, sel_labels = [], []
for c in range(len(CLASS_NAMES)):
    for img in buckets[c]:
        sel_images.append(img)
        sel_labels.append(c)

sel_images = tf.stack(sel_images)
probs = model.predict(sel_images, verbose=0)

ncols = len(CLASS_NAMES)
nrows = PER_CLASS
fig, axes = plt.subplots(nrows, ncols, figsize=(3.5 * ncols, 5.5 * nrows))
fig.patch.set_facecolor("white")
axes = axes.flatten()

for i in range(len(sel_images)):
    ax = axes[i]
    ax.imshow(sel_images[i].numpy().astype("uint8"))
    ax.axis("off")

    vrai = CLASS_NAMES[sel_labels[i]]
    pred_id = int(np.argmax(probs[i]))
    pred = CLASS_NAMES[pred_id]
    ok = (pred_id == sel_labels[i])
    couleur = "green" if ok else "red"

    entete = f"Vrai : {vrai}  |  Predit : {pred}"
    detail = "\n".join(
        f"{CLASS_NAMES[c]:<11} {probs[i][c] * 100:5.1f}%"
        for c in range(len(CLASS_NAMES))
    )

    ax.annotate(
        entete + "\n" + detail,
        xy=(0, 1), xycoords="axes fraction",
        xytext=(0, 12), textcoords="offset points",
        ha="left", va="bottom",
        color=couleur, fontsize=9, family="monospace",
    )

for j in range(len(sel_images), len(axes)):
    axes[j].axis("off")

plt.subplots_adjust(hspace=0.9, wspace=1.1, top=0.45, bottom=0.05)
plt.show()